In [ ]:
!pip install mesa pandas matplotlib


# El Trilema de la Gamarra
## Simulación Multiagente de Informalidad y Evasión Tributaria en el Perú

Modelo MAS con framework **Mesa 3.5** que simula el trilema entre formalidad, evasión 
fiscal y supervivencia microempresarial en Gamarra. Incluye **mecanismos adaptativos 
del mundo real**: bancarrota, entrada dinámica de comerciantes, precios dinámicos, 
SUNAT reactiva y adaptación en Logit.

**Resultados línea base:** 36% formal · 58% evasor · 7% informal (64% informalidad, 
INEI 70.2%, gap −6pp)

**Mecanismos adaptativos:**
- **B1 Bancarrota:** capital < S/500 → cierra
- **B2 Entrada dinámica:** capital formal > S/20,000 → entra nuevo
- **B3 Precios dinámicos:** ±20% según demanda
- **B4 Compliance cost:** si agresividad > 0.80, formal paga 2% extra
- **B5 SUNAT reactiva:** evasión > 60% sostenido → endurece


## Marco Teórico

| Régimen | Tope ingresos | Tasa |
|---|---|---|
| NRUS | 8 UIT/año (S/ 42,800) | Cuota fija S/ 20–50/mes |
| RER | 525 UIT/año | 1.5% sobre ingresos netos |
| RMT | 1,700 UIT/año | 1% hasta 25 UIT, 1.5% después |
| General | Sin tope | 29.5% (Imp. Renta) |

**Causas estructurales de informalidad:**
1. Altos costos de formalidad (IGV 18% + aportes laborales 30% + contabilidad)
2. Enanismo empresarial (98% son microempresas)
3. Baja fiscalización (SUNAT no alcanza millones de microcomerciantes)
4. Baja moral tributaria (consumidores prefieren precios bajos)
5. Pobreza (demanda de bienes baratos)
6. Burocracia y falta de incentivos tangibles


In [ ]:
"""Módulo de parametrización económica, impositiva y laboral. Bienio 2025-2026.

Valores cuantitativos reflejan los decretos supremos vigentes en el Perú.
Fuentes: INEI (EPEN 2025), SUNAT (Código Tributario), MEF (UIT/RMV).
Ver marco teórico en informe_proyecto_gamarra.md.
"""

# Parámetros macroeconómicos y tributarios peruanos
UIT_2025 = 5350.0  # D.S. N° 260-2024-EF
UIT_2026 = 5500.0  # D.S. N° 301-2025-EF
RMV_2025 = 1130.0  # D.S. N° 006-2024-TR

# Selección temporal del simulador: año fiscal 2026
UIT_VIGENTE = UIT_2026
RMV_VIGENTE = RMV_2025
TASA_IGV = 0.18  # 18% (16% IGV + 2% IPM)

# Costos laborales bajo REMYPE microempresa (un trabajador)
COSTO_SIS_EMPLEADOR = 15.00  # SIS Microempresa, pago mensual fijo
VACACIONES_PROVISION = (RMV_VIGENTE / 30.0) * 15.0 / 12.0  # 15 días anuales prorrateados ≈ 47.08
COSTO_LABORAL_MENSUAL = RMV_VIGENTE + COSTO_SIS_EMPLEADOR + VACACIONES_PROVISION  # ≈ 1192.08

# Costo administrativo: contabilidad + trámites mensuales
COSTO_ADMINISTRATIVO_CONTABLE = 200.00

# Costo fijo total mensual de operar formal (promedio: unipersonal + REMYPE con 1 trabajador)
COSTO_FIJO_FORMALIDAD = 400.00  # contabilidad S/200 + NRUS/trámites S/100 + aportes promedio S/100

# Estructura de sanciones (Código Tributario Art. 174)
MULTA_NO_EMISION = 0.50 * UIT_VIGENTE  # S/ 2750 — no emitir comprobante
DESCUENTO_GRADUALIDAD = 0.90  # rebaja 90% por subsanación voluntaria
MULTA_EVASION_PCT = 0.30  # 30% de ingresos ocultos (sanción sobre evasión parcial)

# Comerciante
PRECIO_BASE = 30.0  # precio neto de una prenda (margen comercial Gamarra)
COSTO_UNITARIO = 12.0  # costo de adquisición de materia prima / prenda
CAPITAL_INICIAL = 4000.0  # colchón financiero típico (enanismo: sin escala)
ALPHA_EVASION = 0.60  # fracción de ventas no reportadas por evasor
DISTRIBUCION_INICIAL = ["formal", "evasor", "informal", "informal"]  # 25/25/50

# Consumidor
PRESUPUESTO_MEDIA = 200.0
PRESUPUESTO_DESV = 40.0
MORAL_MIN = 0.05
MORAL_MAX = 0.35  # moral tributaria baja y heterogénea (encuestas)

# Modelo
N_COMERCIANTES = 40
N_CONSUMIDORES = 800  # Gamarra recibe miles de compradores/día — volumen suficiente para viabilidad formal
AGRESIVIDAD_SUNAT = 0.55  # % de comercios auditados por ciclo
TASA_DISCRECIONALIDAD = 0.30  # prob. de acta preventiva vs multa (primera infracción)
SENSIBILIDAD_MERCADO = 3.0  # beta del Logit Multinomial (racionalidad limitada)
ESCALA_LOGIT = 10000.0  # normalización de utilidades para evitar overflow en exp()
PESO_PRECIO = 0.80
PESO_MORAL = 0.20
WIDTH = 15
HEIGHT = 15
SEED = 42

# Mecanismos adaptativos del mundo real
UMBRAL_BANCARROTA = 500.0       # S/. 500 (~10% capital inicial) → comerciante cierra
UMBRAL_CRECIMIENTO = 20000.0    # S/. 20,000 (5x capital inicial) → atrae nuevo entrante
MAX_COMERCIANTES = 80           # tope entrada dinámica
EVASION_ALTA = 60.0             # % evasión que dispara endurecimiento SUNAT
EVASION_BAJA = 50.0             # % formal que relaja SUNAT
SOSTENIMIENTO_FEEDBACK = 20     # ciclos para que SUNAT reaccione
AJUSTE_AGRESIVIDAD_UP = 0.02    # +2pp agresividad/ciclo si evasión alta
AJUSTE_AGRESIVIDAD_DOWN = 0.01  # -1pp agresividad/ciclo si formal alto
MAX_AGRESIVIDAD = 0.95
MIN_AGRESIVIDAD = 0.05
VENTAS_NORMALES = 50            # referencia para precios dinámicos
VENTAS_RANGO = 250              # escala para precios dinámicos
MAX_VARIACION_PRECIO = 0.20     # ±20% sobre PRECIO_BASE por demanda


In [ ]:
"""Comerciante. 3 estrategias: formal / evasor / informal.

Decisión por Logit Multinomial (racionalidad limitada): estima utilidad neta
esperada de cada estrategia y transiciona probabilísticamente.

- Formal: cobra IGV completo, lo declara, paga COSTO_FIJO_FORMALIDAD.
- Evasor: cobra precio formal (con IGV), por cada venta declara o esconde con
  prob alpha. Riesgo: multa proporcional a ingresos ocultos.
- Informal: sin IGV, sin RUC. Riesgo: multa + forzado a evasor si detectado.

B1: Si capital < UMBRAL_BANCARROTA, marca en_quiebra=True. No opera más.
B3: Precio dinámico según ventas del ciclo anterior. Se cachea en step() una vez.
B4: Si agresividad_efectiva > 0.80, costo de cumplimiento del formal sube 2% revenue.

Políticas activables:
- beneficio_antiguedad: descuento incremental en costo_fijo por ciclos formal consecutivo.
- multa_progresiva: el contador de infracciones escala la multa (gestionado por Sunat).
"""
import math
import mesa


class Comerciante(mesa.Agent):
    def __init__(self, model, estrategia: str = "informal", capital: float = CAPITAL_INICIAL):
        super().__init__(model)
        self.estrategia = estrategia
        self.capital = capital
        self.ventas_ciclo = 0
        self.ingresos_declarados = 0.0
        self.ingresos_ocultos = 0.0
        self.multas_pagadas = 0.0
        self.ciclos_formal_consecutivos = 0
        self.infracciones = 0
        # B1: marca de bancarrota
        self.en_quiebra = False
        # B3: precio dinámico (se calcula una vez en step() y se cachea)
        self.precio_actual = PRECIO_BASE

    def _costo_fijo_efectivo(self) -> float:
        """Costo fijo con descuento por antigüedad si la política está activa."""
        costo = self.model.costo_fijo_formalidad
        if self.model.beneficio_antiguedad and self.estrategia == "formal":
            descuento = min(0.50, self.model.tasa_descuento_antiguedad * self.ciclos_formal_consecutivos)
            costo *= (1.0 - descuento)
        return costo

    def _costo_unitario_efectivo(self) -> float:
        """B2: Economías de escala. Si capital > UMBRAL_CRECIMIENTO y formal,
        baja costo unitario hasta 50% (capital muy alto)."""
        if self.capital > UMBRAL_CRECIMIENTO and self.estrategia == "formal":
            reduccion = min(0.50, (self.capital - UMBRAL_CRECIMIENTO) / 100000.0)
            return COSTO_UNITARIO * (1.0 - reduccion)
        return COSTO_UNITARIO

    def _calcular_factor_demanda(self) -> float:
        """B3: factor de demanda según ventas del ciclo anterior.
        Ventas altas → +precio; ventas bajas → -precio. Tope ±MAX_VARIACION_PRECIO.
        """
        delta = (self.ventas_ciclo - VENTAS_NORMALES) / VENTAS_RANGO
        return 1.0 + max(-MAX_VARIACION_PRECIO, min(MAX_VARIACION_PRECIO, delta))

    def calcular_precio(self) -> float:
        """Precio dinámico (B3). precio_actual se cachea en step() una vez por ciclo.

        BUG 18+19 fix: antes recalculaba factor en cada llamada → precio cambiaba
        durante el ciclo y difería entre realizar_compra y ejecutar_transaccion.
        Ahora precio_actual incluye el factor de demanda, calculado una sola vez.
        """
        igv = self.model.tasa_igv
        if self.estrategia in ("formal", "evasor"):
            return self.precio_actual * (1.0 + igv)
        else:  # informal
            return self.precio_actual

    def estimar_utilidad_futura(self, est: str) -> float:
        """Utilidad neta esperada por estrategia (racionalidad limitada).

        Usa revenue real por estrategia (BUG 3 fix) y sin doble descuento de
        IGV (BUG 1 fix). Modelo binario para evasor (BUG 2 fix).
        B4: si agresividad_efectiva > 0.80, costo compliance formal = 2% revenue.
        """
        volumen = max(1, self.ventas_ciclo)
        igv = self.model.tasa_igv
        alpha = self.model.alpha_evasion
        costo_fijo = self.model.costo_fijo_formalidad
        p_inspeccion = self.model.prob_inspeccion_percibida
        disc = self.model.tasa_discrecionalidad
        costos_produccion = self._costo_unitario_efectivo() * volumen  # B2: economías escala

        if est == "formal":
            # Ingreso neto = PRECIO_BASE * volumen (sin IGV, que se paga a SUNAT)
            ingreso_neto = PRECIO_BASE * volumen
            util_antes_imp = ingreso_neto - costos_produccion
            # BUG 11 fix: RMT real 1% primer tramo (no 10%)
            imp_renta = max(0.0, util_antes_imp * 0.01)
            costo_fijo_est = costo_fijo
            if self.model.beneficio_antiguedad:
                descuento = min(0.50, self.model.tasa_descuento_antiguedad * (self.ciclos_formal_consecutivos + 1))
                costo_fijo_est = costo_fijo * (1.0 - descuento)
            # B4: si SUNAT muy agresiva, costo de cumplimiento sube (auditorías, paperwork)
            # proporcional al revenue, no plano — antes era -1000 fijo (bug: siempre negativo)
            compliance_cost = 0.0
            if self.model.agresividad_efectiva > 0.80:
                compliance_cost = ingreso_neto * 0.02  # 2% revenue por audits/paperwork
            return (util_antes_imp - imp_renta) - costo_fijo_est - compliance_cost

        elif est == "informal":
            ingresos_brutos = PRECIO_BASE * volumen  # sin IGV
            # BUG 15 fix: multa solo fija (multa_no_emision), sin 0.50*ingresos
            penalidad = p_inspeccion * self.model.multa_no_emision * (1.0 - disc)
            return ingresos_brutos - costos_produccion - penalidad

        elif est == "evasor":
            # Binario: cobra precio formal, declara (1-alpha) de ventas, esconde alpha
            precio_neto = PRECIO_BASE
            precio_con_igv = PRECIO_BASE * (1.0 + igv)
            # Parte declarada: neto sin IGV
            ing_declarado_neto = (1.0 - alpha) * precio_neto * volumen
            # Parte oculta: precio completo (con IGV cobrado y no remitido)
            ing_oculto_bruto = alpha * precio_con_igv * volumen
            util_declarada = ing_declarado_neto - (costos_produccion * (1.0 - alpha))
            # BUG 11 fix: RMT 1% (no 10%)
            imp_renta = max(0.0, util_declarada * 0.01)
            # BUG 12 fix: alineado con multa real de sunat.py (con factor (1-disc))
            multa_evasion = p_inspeccion * (self.model.multa_evasion_pct * ing_oculto_bruto) * (1.0 - disc)
            util_oculta = ing_oculto_bruto - (costos_produccion * alpha) - multa_evasion
            return (util_declarada - imp_renta - costo_fijo * 0.40) + util_oculta
        return 0.0

    def ajustar_cumplimiento(self) -> None:
        """Transición probabilística vía Logit Multinomial."""
        beta = self.model.sensibilidad_mercado
        escala = self.model.escala_logit
        utilidades = {est: self.estimar_utilidad_futura(est) for est in
                      ("formal", "evasor", "informal")}
        expos = {}
        for est, u in utilidades.items():
            try:
                expos[est] = math.exp(min(500.0, beta * u / escala))
            except OverflowError:
                expos[est] = math.exp(500.0)
        total = sum(expos.values())
        probs = {est: e / total for est, e in expos.items()}

        r = self.random.random()
        acum = 0.0
        for est, p in probs.items():
            acum += p
            if r <= acum:
                if est != self.estrategia:
                    if self.estrategia == "formal":
                        self.ciclos_formal_consecutivos = 0
                self.estrategia = est
                return

    def step(self):
        # B1: bancarrota. Si capital bajo umbral, marca en_quiebra. No opera más.
        if self.en_quiebra:
            return
        if self.capital < UMBRAL_BANCARROTA:
            self.en_quiebra = True
            return
        # B3: setear precio_actual al inicio del ciclo, basado en ventas del ciclo anterior
        # BUG 18 fix: calcular una sola vez (no en cada calcular_precio)
        factor = self._calcular_factor_demanda()
        self.precio_actual = PRECIO_BASE * factor
        self.ajustar_cumplimiento()
        self.ventas_ciclo = 0
        self.ingresos_declarados = 0.0
        self.ingresos_ocultos = 0.0
        if self.estrategia == "formal":
            self.ciclos_formal_consecutivos += 1
            self.capital -= self._costo_fijo_efectivo()
        elif self.estrategia == "evasor":
            self.capital -= self.model.costo_fijo_formalidad * 0.40


In [ ]:
"""Consumidor. Móvil, presupuesto variable, moral tributaria baja.

Elige tienda en vecindario Moore por utilidad precio + moral (comprobante).
La compra enruta el IGV según estrategia del vendedor (F declara, E parcial, I oculto).

Modelo binario para evasor: cobra precio formal, por cada venta decide declarar
(con prob 1-alpha, paga IGV, entra sorteo) o esconder (con prob alpha, oculta).

Política activable:
- sorteo_comprobantes: el consumidor que compra formal (o evasor declarado) tiene
  chance de premio, sumando valor esperado a la utilidad de elegir formal.
"""
import mesa


class Consumidor(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.moral_tributaria = self.random.uniform(MORAL_MIN, MORAL_MAX)
        self.presupuesto = self.random.gauss(PRESUPUESTO_MEDIA, PRESUPUESTO_DESV)

    def _valor_esperado_sorteo(self) -> float:
        """Valor esperado del premio si el modelo tiene sorteo activo."""
        if self.model.sorteo_comprobantes:
            return self.model.prob_sorteo * self.model.premio_sorteo
        return 0.0

    def _prob_comprobante(self, estrategia: str) -> float:
        """Probabilidad esperada de recibir comprobante según estrategia."""
        if estrategia == "formal":
            return 1.0
        elif estrategia == "evasor":
            # BUG 5 fix: evasor solo da comprobante cuando declara (1-alpha)
            return 1.0 - self.model.alpha_evasion
        return 0.0

    def realizar_compra(self) -> None:
        # BUG 9 fix: mantener condición original (permite comprar informal)
        # pero agregar tope anti-loop infinito
        max_movimientos = 20  # ponytail: tope anti-loop infinito
        movimientos = 0
        while self.presupuesto > PRECIO_BASE and movimientos < max_movimientos:
            vecinos = self.model.grid.get_neighbors(self.pos, moore=True, include_center=False)
            comerciantes = [v for v in vecinos if isinstance(v, Comerciante) and not getattr(v, "en_quiebra", False)]
            if not comerciantes:
                self.mover()
                movimientos += 1
                vecinos = self.model.grid.get_neighbors(self.pos, moore=True, include_center=False)
                comerciantes = [v for v in vecinos if isinstance(v, Comerciante) and not getattr(v, "en_quiebra", False)]
                if not comerciantes:
                    return

            mejor = None
            max_util = -float("inf")
            sorteo_ev = self._valor_esperado_sorteo()
            for c in comerciantes:
                precio = c.calcular_precio()
                if precio > self.presupuesto:
                    continue
                # BUG 5 fix: comprobante probabilístico por estrategia
                prob_comp = self._prob_comprobante(c.estrategia)
                # Sorteo: solo si hay probabilidad de comprobante
                bonus_sorteo = sorteo_ev * prob_comp if c.estrategia != "informal" else 0.0
                util = (
                    -(self.model.peso_precio * precio)
                    + (self.model.peso_moral * self.moral_tributaria * prob_comp)
                    + bonus_sorteo
                )
                if util > max_util:
                    max_util = util
                    mejor = c

            if mejor is not None:
                self.ejecutar_transaccion(mejor)
            else:
                self.mover()
                movimientos += 1

    def ejecutar_transaccion(self, comerciante: Comerciante) -> None:
        precio = comerciante.calcular_precio()
        self.presupuesto -= precio
        comerciante.ventas_ciclo += 1
        # BUG 8 fix: comerciante recibe neto (precio - IGV), SUNAT recibe IGV
        igv = self.model.tasa_igv
        alpha = self.model.alpha_evasion

        if comerciante.estrategia == "formal":
            monto_igv = precio * (igv / (1.0 + igv))
            comerciante.capital += precio - monto_igv  # comerciante retiene neto
            comerciante.ingresos_declarados += precio
            self.model.recaudacion_ciclo += monto_igv  # SUNAT recibe IGV
            if self.model.sorteo_comprobantes and self.random.random() < self.model.prob_sorteo:
                # BUG 13 fix: premio sale de recaudación, no de la nada
                self.presupuesto += self.model.premio_sorteo
                self.model.recaudacion_ciclo -= self.model.premio_sorteo
        elif comerciante.estrategia == "evasor":
            # BUG 2 fix: modelo binario — declara o esconde por venta
            if self.random.random() > alpha:
                # Declara esta venta: paga IGV, entra sorteo (BUG 7 fix)
                monto_igv = precio * (igv / (1.0 + igv))
                comerciante.capital += precio - monto_igv  # BUG 8 fix: retiene neto
                comerciante.ingresos_declarados += precio
                self.model.recaudacion_ciclo += monto_igv
                if self.model.sorteo_comprobantes and self.random.random() < self.model.prob_sorteo:
                    # BUG 13 fix: premio sale de recaudación
                    self.presupuesto += self.model.premio_sorteo
                    self.model.recaudacion_ciclo -= self.model.premio_sorteo
            else:
                # Esconde esta venta: no paga IGV, no entra sorteo
                comerciante.capital += precio  # evasor retiene todo (incluido IGV cobrado)
                comerciante.ingresos_ocultos += precio
        else:  # informal
            comerciante.capital += precio
            comerciante.ingresos_ocultos += precio

    def mover(self) -> None:
        pasos = self.model.grid.get_neighborhood(self.pos, moore=True, include_center=False)
        nueva = self.random.choice(pasos)
        self.model.grid.move_agent(self, nueva)

    def step(self):
        self.presupuesto = self.random.gauss(PRESUPUESTO_MEDIA, PRESUPUESTO_DESV)
        self.realizar_compra()


In [ ]:
"""Sunat. Fiscalización con gradualidad y discrecionalidad.

Cobertura = agresividad_sunat (% de comercios auditados/ciclo), leída en vivo
del modelo (BUG 4 fix: antes se capturaba en init y no se actualizaba).

- Informal detectado: multa con gradualidad, forzado a evasor.
- Evasor auditado: multa proporcional a ingresos ocultos, o acta preventiva.

B5: SUNAT reactiva. Si evasión > 60% sostenido 20 ciclos, endurece
(agresividad_efectiva += 0.02). Si formal > 50% sostenido, relaja
(agresividad_efectiva -= 0.01). Modifica model.agresividad_efectiva, no el
slider base (agresividad_sunat).

Contadores per-ciclo (multas_ciclo, actas_ciclo) y acumulados para visualización.

Política activable:
- multa_progresiva: la multa escala según el contador de infracciones del comerciante
  (1ra = acta, 2da = multa base, 3ra+ = multa * factor_reincidencia).
"""
import mesa


class Sunat(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        # Contadores per-ciclo (reset en cada step)
        self.multas_ciclo = 0
        self.actas_ciclo = 0
        self.n_auditorias_ciclo = 0
        # Contadores acumulados (para informe final)
        self.multas_acumuladas = 0
        self.actas_acumuladas = 0
        self.cierres_ejecutados = 0
        # B5: historial de % evasión para feedback adaptativo
        self.historia_evasion = []

    def _factor_reincidencia(self, comerciante: Comerciante) -> float:
        """Factor multiplicador de multa según historial de infracciones."""
        if not self.model.multa_progresiva:
            return 1.0
        if comerciante.infracciones == 0:
            return 0.0  # primera infracción: acta preventiva
        elif comerciante.infracciones == 1:
            return 1.0  # segunda infracción: multa base
        else:
            return self.model.factor_reincidencia ** (comerciante.infracciones - 1)

    def fiscalizar_mercado(self) -> None:
        # Reset contadores per-ciclo
        self.multas_ciclo = 0
        self.actas_ciclo = 0
        self.n_auditorias_ciclo = 0

        comerciantes = [c for c in self.model.agents.select(agent_type=Comerciante)
                        if not getattr(c, "en_quiebra", False)]
        if not comerciantes:
            return
        # BUG 20 fix: usar agresividad_efectiva (B5), no agresividad_sunat (base)
        tasa = self.model.agresividad_efectiva
        n_auditorias = int(len(comerciantes) * tasa)
        muestra = self.random.sample(comerciantes, min(n_auditorias, len(comerciantes)))
        self.n_auditorias_ciclo = len(muestra)

        for c in muestra:
            if c.estrategia == "informal":
                factor = self._factor_reincidencia(c)
                if factor == 0.0 or self.random.random() < self.model.tasa_discrecionalidad:
                    self.actas_ciclo += 1
                    self.actas_acumuladas += 1
                    # BUG 14 fix: acta preventiva no cuenta como infracción
                else:
                    # BUG 15 fix: multa solo fija (multa_no_emision), sin 0.50*ingresos
                    multa = self.model.multa_no_emision * factor
                    # BUG 10 fix: no cobrar si capital negativo
                    cobrado = max(0.0, min(multa, c.capital))
                    c.capital -= cobrado
                    c.multas_pagadas += cobrado
                    self.model.recaudacion_ciclo += cobrado
                    self.multas_ciclo += 1
                    self.multas_acumuladas += 1
                    c.infracciones += 1  # solo multa cuenta como infracción
                c.estrategia = "evasor"

            elif c.estrategia == "evasor":
                factor = self._factor_reincidencia(c)
                if factor == 0.0 or self.random.random() < self.model.tasa_discrecionalidad:
                    self.actas_ciclo += 1
                    self.actas_acumuladas += 1
                    # BUG 14 fix: acta preventiva no cuenta como infracción
                else:
                    # BUG 12 fix: alinear con estimación — aplicar (1-disc) a multa real
                    multa = (c.ingresos_ocultos * self.model.multa_evasion_pct) * factor * (1.0 - self.model.tasa_discrecionalidad)
                    # BUG 10 fix: no cobrar si capital negativo
                    cobrado = max(0.0, min(multa, c.capital))
                    c.capital -= cobrado
                    c.multas_pagadas += cobrado
                    self.model.recaudacion_ciclo += cobrado
                    self.multas_ciclo += 1
                    self.multas_acumuladas += 1
                    c.infracciones += 1  # solo multa cuenta como infracción

        # B5: feedback adaptativo. Lee % evasión actual, actualiza agresividad_efectiva
        self._aplicar_feedback_evasion(comerciantes)

    def _aplicar_feedback_evasion(self, comerciantes: list) -> None:
        """Si evasión alta sostenida → endurece; si formal alto → relaja."""
        if not comerciantes:
            return
        pct_evasion = 100.0 * sum(1 for c in comerciantes
                                   if c.estrategia in ("evasor", "informal")) / len(comerciantes)
        self.historia_evasion.append(pct_evasion)
        if len(self.historia_evasion) > SOSTENIMIENTO_FEEDBACK:
            self.historia_evasion.pop(0)
        if len(self.historia_evasion) < SOSTENIMIENTO_FEEDBACK:
            return
        if all(e > EVASION_ALTA for e in self.historia_evasion):
            self.model.agresividad_efectiva = min(
                MAX_AGRESIVIDAD,
                self.model.agresividad_efectiva + AJUSTE_AGRESIVIDAD_UP
            )
        elif all(e < EVASION_BAJA for e in self.historia_evasion):
            self.model.agresividad_efectiva = max(
                MIN_AGRESIVIDAD,
                self.model.agresividad_efectiva - AJUSTE_AGRESIVIDAD_DOWN
            )

    def step(self):
        self.fiscalizar_mercado()


In [ ]:
"""Modelo MAS principal. Orquesta grilla, step bifásico y DataCollector.

Punto de entrada: python -m src.modelo
"""
import mesa


class ModeloGamarra(mesa.Model):
    def __init__(
        self,
        n_comerciantes: int = N_COMERCIANTES,
        n_consumidores: int = N_CONSUMIDORES,
        agresividad_sunat: float = AGRESIVIDAD_SUNAT,
        tasa_discrecionalidad: float = TASA_DISCRECIONALIDAD,
        peso_precio: float = PESO_PRECIO,
        peso_moral: float = PESO_MORAL,
        sensibilidad_mercado: float = SENSIBILIDAD_MERCADO,
        escala_logit: float = ESCALA_LOGIT,
        costo_fijo_formalidad: float = COSTO_FIJO_FORMALIDAD,
        tasa_igv: float = TASA_IGV,
        multa_no_emision: float = MULTA_NO_EMISION,
        alpha_evasion: float = ALPHA_EVASION,
        multa_evasion_pct: float = MULTA_EVASION_PCT,
        beneficio_antiguedad: bool = False,
        tasa_descuento_antiguedad: float = 0.05,
        sorteo_comprobantes: bool = False,
        prob_sorteo: float = 0.01,
        premio_sorteo: float = 50.0,
        multa_progresiva: bool = False,
        factor_reincidencia: float = 2.0,
        width: int = WIDTH,
        height: int = HEIGHT,
        seed: int = SEED,
    ):
        super().__init__(rng=seed)
        self.grid = mesa.space.MultiGrid(width, height, torus=True)
        self.agresividad_sunat = agresividad_sunat
        # B5: agresividad_efectiva es la que Sunat usa realmente; puede divergir del slider
        # base si el feedback adaptativo la ajusta (evasión alta → endurece, formal alto → relaja)
        self.agresividad_efectiva = agresividad_sunat
        self.tasa_discrecionalidad = tasa_discrecionalidad
        self.sensibilidad_mercado = sensibilidad_mercado
        self.escala_logit = escala_logit
        self.peso_precio = peso_precio
        self.peso_moral = peso_moral
        self.costo_fijo_formalidad = costo_fijo_formalidad
        self.tasa_igv = tasa_igv
        self.multa_no_emision = multa_no_emision
        self.alpha_evasion = alpha_evasion
        self.multa_evasion_pct = multa_evasion_pct
        self.beneficio_antiguedad = beneficio_antiguedad
        self.tasa_descuento_antiguedad = tasa_descuento_antiguedad
        self.sorteo_comprobantes = sorteo_comprobantes
        self.prob_sorteo = prob_sorteo
        self.premio_sorteo = premio_sorteo
        self.multa_progresiva = multa_progresiva
        self.factor_reincidencia = factor_reincidencia
        self.recaudacion_ciclo = 0.0

        # Comerciantes con distribución sesgada a no-formalidad
        for _ in range(n_comerciantes):
            est = self.random.choice(DISTRIBUCION_INICIAL)
            c = Comerciante(self, estrategia=est, capital=CAPITAL_INICIAL)
            self.grid.place_agent(
                c, (self.random.randrange(width), self.random.randrange(height))
            )

        # Consumidores con moral baja heterogénea
        for _ in range(n_consumidores):
            con = Consumidor(self)
            self.grid.place_agent(
                con, (self.random.randrange(width), self.random.randrange(height))
            )

        # Sunat fuera del grid (agente regulador, no espacial)
        self.sunat = Sunat(self)

        self.datacollector = mesa.DataCollector(
            model_reporters={
                # Estrategias (%)
                "pct_formal": lambda m: m._pct("formal"),
                "pct_evasor": lambda m: m._pct("evasor"),
                "pct_informal": lambda m: m._pct("informal"),
                # Recaudación y fiscalización (per-ciclo)
                "recaudacion": lambda m: m.recaudacion_ciclo,
                "multas": lambda m: m.sunat.multas_ciclo,
                "actas_preventivas": lambda m: m.sunat.actas_ciclo,
                "n_auditorias": lambda m: m.sunat.n_auditorias_ciclo,
                # Capital medio por estrategia
                "capital_formal": lambda m: m._capital_medio("formal"),
                "capital_evasor": lambda m: m._capital_medio("evasor"),
                "capital_informal": lambda m: m._capital_medio("informal"),
                # Ventas medias por estrategia
                "ventas_formal": lambda m: m._ventas_medio("formal"),
                "ventas_evasor": lambda m: m._ventas_medio("evasor"),
                "ventas_informal": lambda m: m._ventas_medio("informal"),
                # Ingresos agregados (per-ciclo)
                "ingresos_declarados": lambda m: m._sum_attr_comerciante("ingresos_declarados"),
                "ingresos_ocultos": lambda m: m._sum_attr_comerciante("ingresos_ocultos"),
                # Estado consumidor
                "presupuesto_medio": lambda m: m._consumidor_media("presupuesto"),
                "moral_media": lambda m: m._consumidor_media("moral_tributaria"),
            }
        )

    @property
    def prob_inspeccion_percibida(self) -> float:
        """Los comerciantes estiman el riesgo de inspección del ciclo.

        B5: usa agresividad_efectiva (puede divergir del slider base si SUNAT
        es reactiva).
        """
        return self.agresividad_efectiva

    def _pct(self, estrategia: str) -> float:
        comerciantes = self._comerciantes_activos()
        n = len(comerciantes)
        if not n:
            return 0.0
        return 100.0 * sum(1 for c in comerciantes if c.estrategia == estrategia) / n

    def _capital_medio(self, estrategia: str) -> float:
        comerciantes = [c for c in self._comerciantes_activos()
                        if c.estrategia == estrategia]
        if not comerciantes:
            return 0.0
        return sum(c.capital for c in comerciantes) / len(comerciantes)

    def _ventas_medio(self, estrategia: str) -> float:
        comerciantes = [c for c in self._comerciantes_activos()
                        if c.estrategia == estrategia]
        if not comerciantes:
            return 0.0
        return sum(c.ventas_ciclo for c in comerciantes) / len(comerciantes)

    def _sum_attr_comerciante(self, attr: str) -> float:
        return sum(getattr(c, attr) for c in self._comerciantes_activos())

    def _consumidor_media(self, attr: str) -> float:
        consumidores = self.agents.select(agent_type=Consumidor)
        n = len(consumidores)
        if not n:
            return 0.0
        return sum(getattr(c, attr) for c in consumidores) / n

    def _comerciantes_activos(self):
        """B1: filtra comerciantes en bancarrota."""
        return [c for c in self.agents.select(agent_type=Comerciante)
                if not getattr(c, "en_quiebra", False)]

    def step(self):
        self.recaudacion_ciclo = 0.0
        # Fase 1: mercado (comerciantes ajustan estrategia, consumidores compran)
        self.agents.select(agent_type=Comerciante).shuffle_do("step")
        self.agents.select(agent_type=Consumidor).shuffle_do("step")
        # Fase 2: fiscalización (Sunat lee recaudación y fiscaliza)
        self.sunat.step()
        # B2: entrada de nuevos comerciantes si mercado próspero
        self._entrada_nuevos_comerciantes()
        self.datacollector.collect(self)

    def _entrada_nuevos_comerciantes(self) -> None:
        """B2: si capital medio formal > UMBRAL_CRECIMIENTO y hay espacio, entra nuevo formal."""
        activos = self._comerciantes_activos()
        if len(activos) >= MAX_COMERCIANTES:
            return
        formales = [c for c in activos if c.estrategia == "formal"]
        if not formales:
            return
        capital_medio = sum(c.capital for c in formales) / len(formales)
        if capital_medio > UMBRAL_CRECIMIENTO:
            nuevo = Comerciante(self, estrategia="formal", capital=CAPITAL_INICIAL)
            x, y = self.random.randrange(self.grid.width), self.random.randrange(self.grid.height)
            self.grid.place_agent(nuevo, (x, y))



In [ ]:
import pandas as pd

modelo = ModeloGamarra()
N_CICLOS = 1000
for _ in range(N_CICLOS):
    modelo.step()

df = modelo.datacollector.get_model_vars_dataframe()
print("--- Trayectoria ---")
for i in [0, 100, 500, 999]:
    row = df.iloc[i]
    inf = row["pct_informal"] + row["pct_evasor"]
    print(f"Ciclo {i:4d}: formal {row['pct_formal']:5.1f}%  informal {row['pct_informal']:5.1f}%  "
          f"evasor {row['pct_evasor']:5.1f}%  recaud S/.{row['recaudacion']:7.1f}  "
          f"multas {int(row['multas']):3d}  actas {int(row['actas_preventivas']):3d}")

tail = df.tail(100)
inf_final = tail["pct_informal"].mean() + tail["pct_evasor"].mean()
print("\n--- Promedios últimos 100 ciclos ---")
print(f"Formal:              {tail['pct_formal'].mean():5.1f}%")
print(f"Informal:            {tail['pct_informal'].mean():5.1f}%")
print(f"Evasor:              {tail['pct_evasor'].mean():5.1f}%")
print(f"Recaudación/ciclo:   S/. {tail['recaudacion'].mean():.1f}")
print(f"Multas/ciclo:        {tail['multas'].mean():.1f}")
print(f"Actas prev./ciclo:   {tail['actas_preventivas'].mean():.1f}")
print(f"Auditorías/ciclo:    {tail['n_auditorias'].mean():.1f}")
print(f"Agresividad efectiva: {modelo.agresividad_efectiva:.2f} (base: {modelo.agresividad_sunat})")
print(f"\nInformalidad total (equilibrio): {inf_final:.1f}%  (INEI: ~70%)")
print(f"Gap vs INEI: {inf_final - 70.2:+.1f} pp")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Estrategias
ax = axes[0, 0]
ax.plot(df.index, df["pct_formal"], color="green", label="Formal")
ax.plot(df.index, df["pct_evasor"], color="orange", label="Evasor")
ax.plot(df.index, df["pct_informal"], color="red", label="Informal")
ax.set_title("Estrategias de comerciantes (%)")
ax.set_xlabel("Ciclo")
ax.legend()
ax.grid(True, alpha=0.3)

# Recaudación
ax = axes[0, 1]
ax.plot(df.index, df["recaudacion"], color="steelblue")
ax.set_title("Recaudación por ciclo (S/.)")
ax.set_xlabel("Ciclo")
ax.grid(True, alpha=0.3)

# Multas y actas
ax = axes[1, 0]
ax.plot(df.index, df["multas"], color="firebrick", label="Multas")
ax.plot(df.index, df["actas_preventivas"], color="navy", label="Actas")
ax.set_title("Multas y actas preventivas por ciclo")
ax.set_xlabel("Ciclo")
ax.legend()
ax.grid(True, alpha=0.3)

# Agresividad efectiva
ax = axes[1, 1]
ax.plot(df.index, df["n_auditorias"], color="purple")
ax.set_title("Auditorías por ciclo")
ax.set_xlabel("Ciclo")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
valores = [0.05, 0.25, 0.55, 0.80, 0.95]
resultados = []

for a in valores:
    m = ModeloGamarra(agresividad_sunat=a)
    for _ in range(1000):
        m.step()
    df_sweep = m.datacollector.get_model_vars_dataframe()
    tail = df_sweep.tail(100)
    resultados.append({
        "agresividad": a,
        "formal": tail["pct_formal"].mean(),
        "evasor": tail["pct_evasor"].mean(),
        "informal": tail["pct_informal"].mean(),
        "recaudacion": tail["recaudacion"].mean(),
        "agres_efectiva": m.agresividad_efectiva,
    })

sweep_df = pd.DataFrame(resultados)
print(sweep_df.to_string(index=False))


In [ ]:
escenarios = {
    "Línea base": {},
    "Más enforcement": {"agresividad_sunat": 0.90, "tasa_discrecionalidad": 0.10, "multa_no_emision": 5000, "multa_evasion_pct": 1.0},
    "Subsidio formalidad": {"costo_fijo_formalidad": 150, "beneficio_antiguedad": True, "tasa_descuento_antiguedad": 0.08},
    "Cultura tributaria": {"peso_moral": 0.40, "sorteo_comprobantes": True, "prob_sorteo": 0.03, "premio_sorteo": 100},
    "Reducción tributaria": {"tasa_igv": 0.10, "alpha_evasion": 0.30},
    "Combo reforma": {"agresividad_sunat": 0.75, "costo_fijo_formalidad": 200, "peso_moral": 0.35, "beneficio_antiguedad": True, "sorteo_comprobantes": True, "multa_progresiva": True},
    "Paraíso formal": {"costo_fijo_formalidad": 100, "tasa_igv": 0.05, "peso_moral": 0.50, "alpha_evasion": 0.10, "beneficio_antiguedad": True, "sorteo_comprobantes": True},
    "Mercado libre": {"agresividad_sunat": 0.05, "multa_no_emision": 500, "multa_evasion_pct": 0.1},
    "Represión extrema": {"agresividad_sunat": 0.95, "multa_no_emision": 5500, "tasa_discrecionalidad": 0.0, "multa_evasion_pct": 1.5},
    "Crisis de demanda": {"n_consumidores": 200, "costo_fijo_formalidad": 800},
    "Evasión incentivada": {"alpha_evasion": 0.90, "multa_evasion_pct": 0.1, "agresividad_sunat": 0.10},
}

resultados_esc = []
for nombre, params in escenarios.items():
    m = ModeloGamarra(**params)
    for _ in range(1000):
        m.step()
    tail = m.datacollector.get_model_vars_dataframe().tail(100)
    f = tail["pct_formal"].mean()
    e = tail["pct_evasor"].mean()
    i = tail["pct_informal"].mean()
    resultados_esc.append({
        "escenario": nombre,
        "formal": f"{f:.0f}%",
        "evasor": f"{e:.0f}%",
        "informal": f"{i:.0f}%",
        "informal_total": f"{e+i:.0f}%",
        "recaudacion": f"S/. {tail['recaudacion'].mean():.0f}",
        "agres_efectiva": f"{m.agresividad_efectiva:.2f}",
    })

esc_df = pd.DataFrame(resultados_esc)
print(esc_df.to_string(index=False))


## Hallazgos

### El trilema estructural
El regulador **no puede maximizar simultáneamente** recaudación, estándares formales 
y estabilidad del empleo microempresarial. El modelo reproduce esto: incluso con 
reformas, el evasor persiste como estrategia racional dominante.

### SUNAT reactiva (B5) es poderosa
La agresividad efectiva sube automáticamente cuando la evasión supera 60% sostenido. 
En la línea base, SUNAT endurece de 0.55 a 0.77. Esto duplica las auditorías y 
triplica la recaudación por multas.

### Bancarrota y entrada dinámica
Los comerciantes formales con capital < S/500 quiebran (B1), pero si el mercado 
es próspero, entran nuevos (B2). El número de comerciantes oscila entre 40 y 80.

### Precios dinámicos
Los comerciantes ajustan precios ±20% según demanda. Ventas altas → suben precio; 
ventas bajas → bajan para captar mercado.

### Represión extrema: formal pero destruido
Con enforcement máximo, formal llega a 51% PERO con 9 quiebras. El mercado se 
destruye por exceso de fiscalización — no es solución, es demolición.


## Recomendaciones de Política Pública

1. **Sustituir multas por capacitación voluntaria** — actas preventivas en primera 
   infracción reducen destrucción de capital sin eliminar señalización.
2. **Atenuar brecha costo laboral** — cofinanciar aportes previsionales primeros 
   3 años de REMYPE. Reducir `costo_fijo_formalidad`.
3. **Simplificar regímenes tributarios** — fusionar RER y RMT en régimen progresivo 
   único, eliminar topes rígidos que incentivan fragmentación.
4. **Estimular moral tributaria** — fortalecer sorteos de comprobantes. Elevar 
   `peso_moral` de 0.20 a 0.40 reduce informalidad más que doblar enforcement.

**Ninguna palanca aislada basta.** Reducir informalidad al 50% (meta gobierno) 
requiere actuar en enforcement + costos + cultura tributaria simultáneamente.


## Dashboard Interactivo (Streamlit)

El dashboard completo corre con Streamlit (más estable que Solara):

```bash
streamlit run src/visualizacion.py
```

Incluye: grid visual, gráfica de convergencia al equilibrio, panel de implicaciones 
con zonas temporales, métricas en vivo, banner de estado, 12 escenarios preset.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Grid visual
ax1.set_aspect("equal")
ax1.set_xticks([])
ax1.set_yticks([])
ax1.set_title("Grid del mercado — estado final")
colores = {"formal": "green", "evasor": "orange", "informal": "red"}
for agent in modelo.agents:
    if not hasattr(agent, "pos") or agent.pos is None:
        continue
    x, y = agent.pos
    if isinstance(agent, Comerciante):
        if getattr(agent, "en_quiebra", False):
            ax1.scatter(x, y, c="gray", s=60, marker="x", alpha=0.3)
        else:
            c = colores.get(agent.estrategia, "gray")
            ax1.scatter(x, y, c=c, s=120, marker="s", edgecolors="black", linewidths=0.5)
    elif isinstance(agent, Sunat):
        ax1.scatter(x, y, c="black", s=150, marker="X")
    elif isinstance(agent, Consumidor):
        ax1.scatter(x, y, c="royalblue", s=10, marker=".", alpha=0.15)

handles = [
    mpatches.Patch(color="green", label="Formal"),
    mpatches.Patch(color="orange", label="Evasor"),
    mpatches.Patch(color="red", label="Informal"),
    mpatches.Patch(color="gray", label="En quiebra"),
    plt.Line2D([0], [0], marker="X", color="w", markerfacecolor="black", markersize=10, label="SUNAT"),
]
ax1.legend(handles=handles, loc="upper left", fontsize=7)

# Convergencia
ax2.plot(df.index, df["pct_formal"], color="green", label="Formal")
ax2.plot(df.index, df["pct_evasor"], color="orange", label="Evasor")
ax2.plot(df.index, df["pct_informal"], color="red", label="Informal")
ax2.set_title("Convergencia al equilibrio")
ax2.set_xlabel("Ciclo")
ax2.set_ylabel("%")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
